# Notebook 13: Blood Transcriptomics with ADOS Clinical Correlation

**Dataset:** [GSE111175](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE111175) — Gazestani et al. 2019, *Nature Neuroscience*
**Title:** "A perturbed gene network containing PI3K-AKT, RAS-ERK and WNT-beta-catenin pathways in leukocytes is linked to ASD genetics and symptom severity"
**Platform:** Illumina HumanHT-12 V4.0 Expression BeadChip (GPL10558)
**Samples:** ~254 blood leukocyte samples from toddlers aged 1-4 (128 ASD + 126 controls)
**Clinical data:** ADOS scores (Social Affect, Communication), Mullen Scales, Vineland Scores

**Framework:** [pathway-subtyping](https://pypi.org/project/pathway-subtyping/) v0.3.0
**Author:** Rohit Chauhan ([ORCID: 0009-0003-9895-4629](https://orcid.org/0009-0003-9895-4629))

---

## What this notebook does

1. Downloads GSE111175 blood expression data from GEO
2. Extracts clinical metadata including ADOS severity scores
3. Runs pathway-level scoring using 15 curated autism pathways
4. Discovers molecular subtypes via GMM clustering
5. Validates subtypes through 3 validation gates
6. Characterizes subtypes (enriched pathways, top genes)
7. **Tests clinical correlation: Do subtypes differ in ADOS severity?** (KEY ANALYSIS)
8. Compares blood subtypes to postmortem brain subtypes (GSE28521)
9. Benchmarks against alternative clustering methods

**Vulnerabilities addressed:**
- **V5** — Sample size (n~254 vs n=32 in GSE28521)
- **V6** — No clinical phenotype correlation (ADOS analysis)
- **V7** — Postmortem brain only (blood-based subtyping)

**Runtime:** ~10-15 minutes on Colab Pro

## 1. Setup & Installation

In [ ]:
# Install pathway-subtyping framework with visualization extras
!pip install -q pathway-subtyping[viz]==0.3.0 GEOparse

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Framework imports
from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
    run_clustering,
    ClusteringAlgorithm,
    select_n_clusters,
    compare_algorithms,
    ValidationGates,
    characterize_subtypes,
    generate_subtype_heatmap,
    generate_gene_heatmap,
    export_characterization,
    run_benchmark_comparison,
    compute_dim_reduction,
    DimReductionMethod,
)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directory
OUTPUT_DIR = './outputs/gse111175'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Setup complete.')

## 2. Download GSE111175 from GEO

We use GEOparse to download the series matrix file containing probe-level
expression values and sample metadata including ADOS clinical scores.

In [ ]:
import GEOparse

print('Downloading GSE111175 from GEO (this may take 1-2 minutes)...')
gse = GEOparse.get_GEO(geo='GSE111175', destdir=DATA_DIR, silent=True)
print(f'Downloaded. Platform(s): {list(gse.gpls.keys())}')
print(f'Number of samples: {len(gse.gsms)}')

## 3. Extract Sample Metadata & Clinical Scores

Parse sample characteristics to extract diagnosis, age, sex, and critically,
**ADOS scores** (Autism Diagnostic Observation Schedule) for clinical correlation.

In [ ]:
# Extract all metadata from sample characteristics
metadata_rows = []
for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get('characteristics_ch1', [])
    char_dict = {}
    for c in chars:
        if ':' in c:
            key, val = c.split(':', 1)
            char_dict[key.strip().lower()] = val.strip()

    title = gsm.metadata.get('title', [''])[0]
    source = gsm.metadata.get('source_name_ch1', [''])[0]

    metadata_rows.append({
        'sample_id': gsm_name,
        'title': title,
        'source': source,
        **char_dict,
    })

metadata = pd.DataFrame(metadata_rows).set_index('sample_id')
print(f'Metadata columns: {list(metadata.columns)}')
print(f'Total samples: {len(metadata)}')
metadata.head()

In [ ]:
# Parse diagnosis from metadata
# The diagnosis column may be named differently — inspect and adapt
diag_col = None
for col in metadata.columns:
    if 'diagnosis' in col.lower() or 'disease' in col.lower() or 'status' in col.lower() or 'group' in col.lower():
        diag_col = col
        break

if diag_col is None:
    # Try source column or title
    print('No explicit diagnosis column found. Checking source/title...')
    print(f'Unique sources: {metadata["source"].unique()[:10]}')
    print(f'Sample titles: {metadata["title"].unique()[:5]}')
    # Try to infer from source
    if metadata['source'].str.contains('ASD|autism', case=False).any():
        metadata['diagnosis'] = metadata['source'].apply(
            lambda x: 'ASD' if 'ASD' in x.upper() or 'autism' in x.lower() else 'Control'
        )
        diag_col = 'diagnosis'
else:
    metadata['diagnosis'] = metadata[diag_col].apply(
        lambda x: 'ASD' if 'asd' in str(x).lower() or 'autism' in str(x).lower() else 'Control'
    )

print(f'\nDiagnosis column: {diag_col}')
print(f'\n--- Sample Breakdown ---')
print(metadata['diagnosis'].value_counts())

In [ ]:
# Extract ADOS scores and other clinical variables
# Look for ADOS-related columns
ados_cols = [c for c in metadata.columns if 'ados' in c.lower()]
clinical_cols = [c for c in metadata.columns if any(kw in c.lower() for kw in
                 ['ados', 'age', 'sex', 'gender', 'mullen', 'vineland'])]

print(f'ADOS columns found: {ados_cols}')
print(f'All clinical columns: {clinical_cols}')

# Convert ADOS scores to numeric
for col in ados_cols:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')

# Convert age to numeric
age_cols = [c for c in metadata.columns if 'age' in c.lower()]
for col in age_cols:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')

# Show clinical data summary for ASD samples
asd_mask = metadata['diagnosis'] == 'ASD'
print(f'\n--- Clinical Summary (ASD samples, n={asd_mask.sum()}) ---')
for col in clinical_cols:
    if metadata[col].dtype in ['float64', 'int64']:
        vals = metadata.loc[asd_mask, col].dropna()
        print(f'  {col}: n={len(vals)}, mean={vals.mean():.2f}, std={vals.std():.2f}, '
              f'range=[{vals.min():.1f}, {vals.max():.1f}]')
    else:
        print(f'  {col}: {metadata.loc[asd_mask, col].value_counts().to_dict()}')

# Identify the primary ADOS score column for later use
# Prefer ADOS Social Affect (ADOS-SA) or total ADOS
ADOS_COL = None
for candidate in ['ados_social_affect', 'ados sa', 'ados-sa', 'ados total', 'ados']:
    matches = [c for c in ados_cols if candidate in c.lower().replace('_', ' ')]
    if matches:
        ADOS_COL = matches[0]
        break
if ADOS_COL is None and ados_cols:
    ADOS_COL = ados_cols[0]  # fallback to first ADOS column

print(f'\nPrimary ADOS column for analysis: {ADOS_COL}')
if ADOS_COL:
    print(f'  Non-null values: {metadata[ADOS_COL].notna().sum()}')
    print(f'  ASD with ADOS: {metadata.loc[asd_mask, ADOS_COL].notna().sum()}')

## 4. Build Expression Matrix

Extract probe-level expression values from the GEO series matrix,
then map probes to gene symbols using the Illumina HumanHT-12 V4.0 platform annotation.

In [ ]:
# Extract expression table
expression_df = gse.pivot_samples('VALUE')
print(f'Raw expression matrix: {expression_df.shape[0]} probes x {expression_df.shape[1]} samples')

# Ensure numeric
expression_df = expression_df.apply(pd.to_numeric, errors='coerce')
expression_df = expression_df.dropna(how='all')
print(f'After dropping all-NaN probes: {expression_df.shape[0]} probes')

In [ ]:
# Map probes to gene symbols using platform annotation
gpl = list(gse.gpls.values())[0]
gpl_table = gpl.table

# Find the gene symbol column
symbol_col = None
for col in ['Symbol', 'Gene Symbol', 'GENE_SYMBOL', 'Gene_Symbol', 'ILMN_Gene']:
    if col in gpl_table.columns:
        symbol_col = col
        break

if symbol_col is None:
    for col in gpl_table.columns:
        if 'symbol' in col.lower() or 'gene' in col.lower():
            symbol_col = col
            break

print(f'Platform: {gpl.metadata.get("title", ["Unknown"])[0]}')
print(f'Using gene symbol column: "{symbol_col}"')

# Create probe-to-gene mapping
probe_to_gene = gpl_table.set_index('ID')[symbol_col].dropna()
probe_to_gene = probe_to_gene[probe_to_gene.str.strip() != '']
print(f'Probes with gene symbols: {len(probe_to_gene)}')

In [ ]:
# Map probes to genes and collapse (mean of probes per gene)
common_probes = expression_df.index.intersection(probe_to_gene.index)
expression_mapped = expression_df.loc[common_probes].copy()
expression_mapped['gene_symbol'] = probe_to_gene.loc[common_probes].values

# Collapse multiple probes per gene by taking the mean
gene_expression = expression_mapped.groupby('gene_symbol').mean()
print(f'Gene-level expression: {gene_expression.shape[0]} genes x {gene_expression.shape[1]} samples')

# Transpose to samples x genes
gene_expression = gene_expression.T
print(f'Transposed: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes')

# Check if already log-transformed
max_val = gene_expression.max().max()
print(f'Max expression value: {max_val:.2f}')
if max_val > 30:
    print('Data appears to be in raw scale — applying log2(x+1) transform')
    gene_expression = np.log2(gene_expression + 1)
else:
    print('Data appears to be already log-transformed — no additional transform needed')

In [ ]:
# Basic QC
print('--- Expression Matrix QC ---')
print(f'Shape: {gene_expression.shape}')
print(f'Missing values: {gene_expression.isna().sum().sum()}')
print(f'Expression range: [{gene_expression.min().min():.2f}, {gene_expression.max().max():.2f}]')
print(f'Mean expression: {gene_expression.mean().mean():.2f}')

# Drop genes with zero variance
gene_var = gene_expression.var()
n_zero_var = (gene_var == 0).sum()
if n_zero_var > 0:
    gene_expression = gene_expression.loc[:, gene_var > 0]
    print(f'Dropped {n_zero_var} zero-variance genes. Remaining: {gene_expression.shape[1]}')

# Fill any remaining NaN with column median
if gene_expression.isna().any().any():
    gene_expression = gene_expression.fillna(gene_expression.median())
    print('Filled remaining NaN values with column medians.')

# Align metadata index with expression index
common_samples = gene_expression.index.intersection(metadata.index)
gene_expression = gene_expression.loc[common_samples]
metadata = metadata.loc[common_samples]
print(f'\nFinal: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes')
print(f'Metadata aligned: {len(metadata)} samples')

## 5. Load Autism Pathway Gene Sets & Score

The framework ships with 15 curated autism pathway gene sets derived from
SFARI Gene, Satterstrom et al. 2020, and ASC exome studies.

We use ssGSEA (single-sample Gene Set Enrichment Analysis) to reduce
the ~20,000 gene expression matrix to a 15-pathway score matrix.

In [ ]:
import urllib.request

GMT_URL = 'https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/data/pathways/autism_pathways.gmt'
GMT_PATH = os.path.join(DATA_DIR, 'autism_pathways.gmt')

if not os.path.exists(GMT_PATH):
    urllib.request.urlretrieve(GMT_URL, GMT_PATH)
    print(f'Downloaded autism_pathways.gmt')

# Parse GMT file
pathways = {}
with open(GMT_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = line.split('\t')
        if len(parts) >= 3:
            pathways[parts[0]] = parts[2:]

print(f'Loaded {len(pathways)} pathways:')
total_genes = set()
for name, genes in pathways.items():
    available = len(set(genes) & set(gene_expression.columns))
    total_genes.update(genes)
    print(f'  {name}: {len(genes)} genes ({available} found in expression data)')

print(f'\nTotal unique pathway genes: {len(total_genes)}')
print(f'Found in expression data: {len(total_genes & set(gene_expression.columns))}')

In [ ]:
# Score pathways using ssGSEA
scoring_result = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores = scoring_result.pathway_scores

print('\n--- Scoring Report ---')
print(scoring_result.format_report())
print(f'\nPathway score matrix: {pathway_scores.shape}')
print(f'Pathways scored: {scoring_result.n_pathways_scored}')
if scoring_result.skipped_pathways:
    print(f'Skipped: {scoring_result.skipped_pathways}')

In [ ]:
# Visualize pathway score distribution by diagnosis
scores_with_meta = pathway_scores.copy()
scores_with_meta['diagnosis'] = metadata.loc[pathway_scores.index, 'diagnosis']

n_pathways = pathway_scores.shape[1]
n_cols = 5
n_rows = (n_pathways + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()

for i, pathway in enumerate(pathway_scores.columns):
    ax = axes_flat[i]
    for dx, color in [('ASD', 'coral'), ('Control', 'steelblue')]:
        subset = scores_with_meta[scores_with_meta['diagnosis'] == dx][pathway]
        ax.hist(subset, alpha=0.6, label=dx, color=color, bins=15)
    ax.set_title(pathway.replace('_', '\n'), fontsize=8)
    if i == 0:
        ax.legend(fontsize=7)

# Hide unused axes
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('GSE111175 Blood: Pathway Score Distributions (ASD vs Control)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pathway_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Optimal Cluster Selection

Use BIC (Bayesian Information Criterion) to select the optimal number of molecular subtypes.
We test k=2 through k=7 on the ASD samples only — controls serve as a reference population.

In [ ]:
# Subset to ASD samples only for subtype discovery
asd_mask = metadata['diagnosis'] == 'ASD'
asd_scores = pathway_scores.loc[asd_mask]
asd_expression = gene_expression.loc[asd_mask]
asd_meta = metadata.loc[asd_mask].copy()

print(f'ASD samples for subtype discovery: {len(asd_scores)}')
print(f'Control samples (reference): {(~asd_mask).sum()}')

# Select optimal number of clusters using BIC
selection = select_n_clusters(
    data=asd_scores.values,
    k_range=list(range(2, 8)),
    method='bic',
    seed=SEED,
)

optimal_k = selection.optimal_k
print(f'\nOptimal k (BIC): {optimal_k}')
print(f'\nBIC values: {selection.bic_values}')
print(f'Silhouette values: {selection.silhouette_values}')

In [ ]:
# Plot BIC and Silhouette curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ks = sorted(selection.bic_values.keys())
ax1.plot(ks, [selection.bic_values[k] for k in ks], 'bo-', linewidth=2)
ax1.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('BIC (lower is better)')
ax1.set_title('Model Selection: BIC')
ax1.legend()

ax2.plot(ks, [selection.silhouette_values[k] for k in ks], 'go-', linewidth=2)
ax2.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score (higher is better)')
ax2.set_title('Model Selection: Silhouette')
ax2.legend()

plt.suptitle('GSE111175 Blood (ASD only): Optimal Cluster Selection',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_selection.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. GMM Clustering & Visualization

Run Gaussian Mixture Model clustering at the BIC-optimal k on ASD samples.
Then visualize subtypes in PCA space alongside diagnosis and clinical metadata.

In [ ]:
# Run GMM clustering on ASD samples
clustering = run_clustering(
    data=asd_scores.values,
    n_clusters=optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)

print(f'--- GMM Clustering Results (ASD only) ---')
print(f'k = {clustering.n_clusters}')
print(f'Silhouette score: {clustering.silhouette:.4f}')
print(f'Calinski-Harabasz: {clustering.calinski_harabasz:.2f}')
print(f'Davies-Bouldin: {clustering.davies_bouldin:.4f}')
if clustering.bic is not None:
    print(f'BIC: {clustering.bic:.2f}')
print(f'Converged: {clustering.converged}')

# Assign labels
labels = clustering.labels
asd_meta['subtype'] = labels

print(f'\nSubtype sizes:')
for i in range(optimal_k):
    count = (labels == i).sum()
    print(f'  Subtype {i}: {count} samples ({count/len(labels)*100:.1f}%)')

In [ ]:
# Cross-tabulate subtypes with available clinical metadata
print('--- Subtype Distribution ---')
print(pd.Series(labels).value_counts().sort_index())

# If sex/gender column exists, cross-tabulate
sex_col = None
for col in asd_meta.columns:
    if 'sex' in col.lower() or 'gender' in col.lower():
        sex_col = col
        break

if sex_col:
    print(f'\n--- Subtype × Sex ---')
    print(pd.crosstab(asd_meta['subtype'], asd_meta[sex_col], margins=True))

# Age distribution by subtype
age_col = None
for col in asd_meta.columns:
    if 'age' in col.lower():
        age_col = col
        break

if age_col and asd_meta[age_col].notna().sum() > 0:
    print(f'\n--- Age by Subtype ---')
    for s in sorted(asd_meta['subtype'].unique()):
        ages = asd_meta.loc[asd_meta['subtype'] == s, age_col].dropna()
        if len(ages) > 0:
            print(f'  Subtype {s}: mean={ages.mean():.2f}, std={ages.std():.2f}, n={len(ages)}')

In [ ]:
# PCA scatter plot colored by subtype and diagnosis
embedding, pca_meta = compute_dim_reduction(
    pathway_scores=asd_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Color by subtype
scatter_colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    mask = labels == i
    axes[0].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                    label=f'Subtype {i} (n={mask.sum()})', s=60, alpha=0.7,
                    edgecolors='k', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca_meta["explained_variance_ratio"][0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_meta["explained_variance_ratio"][1]*100:.1f}%)')
axes[0].set_title('ASD Blood Subtypes')
axes[0].legend()

# Right: Color by ADOS score (if available)
if ADOS_COL and asd_meta[ADOS_COL].notna().sum() > 5:
    ados_vals = asd_meta.loc[asd_scores.index, ADOS_COL].values
    valid = ~np.isnan(ados_vals)
    sc = axes[1].scatter(embedding[valid, 0], embedding[valid, 1], c=ados_vals[valid],
                         cmap='YlOrRd', s=60, alpha=0.8, edgecolors='k', linewidth=0.5)
    plt.colorbar(sc, ax=axes[1], label=ADOS_COL.replace('_', ' ').title())
    # Mark samples without ADOS scores
    if (~valid).sum() > 0:
        axes[1].scatter(embedding[~valid, 0], embedding[~valid, 1], c='gray',
                        marker='x', s=40, alpha=0.5, label='No ADOS')
        axes[1].legend(fontsize=8)
    axes[1].set_xlabel(f'PC1 ({pca_meta["explained_variance_ratio"][0]*100:.1f}%)')
    axes[1].set_ylabel(f'PC2 ({pca_meta["explained_variance_ratio"][1]*100:.1f}%)')
    axes[1].set_title(f'Colored by {ADOS_COL.replace("_", " ").title()}')
else:
    # Fallback: just repeat subtype coloring with different style
    for i in range(optimal_k):
        mask = labels == i
        axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                        label=f'Subtype {i}', s=60, alpha=0.7, marker='s',
                        edgecolors='k', linewidth=0.5)
    axes[1].set_title('Subtype Assignments')
    axes[1].legend()

plt.suptitle(f'GSE111175 Blood: Pathway-Based Molecular Subtypes (k={optimal_k}, GMM)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pca_scatter_subtypes.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Validation Gates

Run the framework's validation gates to confirm the subtypes are biologically meaningful:
1. **Negative Control 1 (Label Shuffle):** Shuffled labels should NOT be recoverable
2. **Negative Control 2 (Random Gene Sets):** Random pathways should NOT reproduce the clusters
3. **Stability (Bootstrap):** Clusters should survive resampling

In [ ]:
# Run all validation gates
gates = ValidationGates(
    seed=SEED,
    n_permutations=200,
    n_bootstrap=100,
    stability_threshold=0.8,
    null_ari_max=0.15,
    show_progress=True,
)

print('Running validation gates (this may take 1-2 minutes)...')
val_result = gates.run_all(
    pathway_scores=asd_scores,
    cluster_labels=labels,
    pathways=pathways,
    gene_burdens=asd_expression,
    n_clusters=optimal_k,
    gmm_seed=SEED,
)

print('\n' + '=' * 60)
print('VALIDATION GATES RESULTS')
print('=' * 60)
print(f'\nAll gates passed: {"YES" if val_result.all_passed else "NO"}')
print()
for gate in val_result.results:
    status = 'PASS' if gate.passed else 'FAIL'
    print(f'  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} '
          f'(threshold: {gate.comparison} {gate.threshold:.4f})')

## 9. Subtype Characterization

Identify which pathways and genes drive each molecular subtype.
This reveals the biological signature of each blood-based ASD subtype.

In [ ]:
# Characterize subtypes
char_result = characterize_subtypes(
    pathway_scores=asd_scores,
    cluster_labels=labels,
    gene_burdens=asd_expression,
    pathways=pathways,
    fdr_alpha=0.05,
    top_n_genes=20,
    seed=SEED,
)

# Print characterization report
print(char_result.format_report())

In [ ]:
# Generate heatmaps
fig_heatmap = generate_subtype_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, 'subtype_heatmap.png'),
    figsize=(14, 8),
)
plt.show()

fig_genes = generate_gene_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, 'gene_heatmap.png'),
    figsize=(16, 10),
    top_n=15,
)
plt.show()

In [ ]:
# Export characterization data to CSV
export_files = export_characterization(
    char_result,
    output_dir=OUTPUT_DIR,
    formats=['csv'],
)
print('Exported characterization files:')
for f in export_files:
    print(f'  {f}')

## 10. ADOS Clinical Correlation — KEY ANALYSIS

**This is the central analysis addressing Vulnerability V6** (no clinical phenotype correlation).

We test whether the blood-based molecular subtypes discovered in Section 7 correspond to
meaningful differences in clinical severity as measured by the Autism Diagnostic Observation
Schedule (ADOS).

**Statistical tests:**
- **Kruskal-Wallis H test:** Do subtypes differ in ADOS scores overall?
- **Mann-Whitney U tests:** Pairwise subtype comparisons with Bonferroni correction
- **Cohen's d:** Effect size for each pairwise comparison
- **Spearman correlation:** Pathway scores vs ADOS severity (continuous relationship)

**Significance threshold:** p < 0.05 (corrected for multiple comparisons)

In [ ]:
from scipy import stats

# Gather all available ADOS/clinical score columns for testing
test_cols = [c for c in ados_cols if asd_meta[c].notna().sum() >= 10]

# Also include any Mullen/Vineland scores
for col in metadata.columns:
    if any(kw in col.lower() for kw in ['mullen', 'vineland']) and col not in test_cols:
        metadata[col] = pd.to_numeric(metadata[col], errors='coerce')
        asd_meta[col] = metadata.loc[asd_meta.index, col]
        if asd_meta[col].notna().sum() >= 10:
            test_cols.append(col)

print(f'Clinical score columns available for testing (n >= 10): {test_cols}')
print()

# Kruskal-Wallis test for each clinical score
kw_results = []
for col in test_cols:
    groups = []
    for s in sorted(asd_meta['subtype'].unique()):
        vals = asd_meta.loc[asd_meta['subtype'] == s, col].dropna().values
        if len(vals) >= 3:
            groups.append(vals)

    if len(groups) >= 2:
        h_stat, p_val = stats.kruskal(*groups)
        kw_results.append({'score': col, 'H_statistic': h_stat, 'p_value': p_val,
                           'n_groups': len(groups), 'total_n': sum(len(g) for g in groups)})
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        print(f'{col}: H={h_stat:.3f}, p={p_val:.4f} {sig} (n={sum(len(g) for g in groups)})')
    else:
        print(f'{col}: insufficient data for testing')

kw_df = pd.DataFrame(kw_results)
if len(kw_df) > 0:
    kw_df['significant'] = kw_df['p_value'] < 0.05
    print(f'\n--- Summary ---')
    print(f'Scores tested: {len(kw_df)}')
    print(f'Significant (p<0.05): {kw_df["significant"].sum()}')

In [ ]:
# Pairwise Mann-Whitney U tests with Bonferroni correction for ADOS column
from itertools import combinations

if ADOS_COL and ADOS_COL in test_cols:
    primary_col = ADOS_COL
elif test_cols:
    primary_col = test_cols[0]
else:
    primary_col = None

if primary_col:
    print(f'=== Pairwise Comparisons: {primary_col} ===\n')

    subtypes = sorted(asd_meta['subtype'].unique())
    pairs = list(combinations(subtypes, 2))
    n_comparisons = len(pairs)

    pairwise_results = []
    for s1, s2 in pairs:
        g1 = asd_meta.loc[asd_meta['subtype'] == s1, primary_col].dropna().values
        g2 = asd_meta.loc[asd_meta['subtype'] == s2, primary_col].dropna().values

        if len(g1) >= 3 and len(g2) >= 3:
            u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative='two-sided')
            p_corrected = min(p_val * n_comparisons, 1.0)  # Bonferroni

            # Cohen's d
            pooled_std = np.sqrt((np.std(g1, ddof=1)**2 + np.std(g2, ddof=1)**2) / 2)
            cohens_d = (np.mean(g1) - np.mean(g2)) / pooled_std if pooled_std > 0 else 0

            sig = '***' if p_corrected < 0.001 else '**' if p_corrected < 0.01 else '*' if p_corrected < 0.05 else 'ns'
            print(f'  Subtype {s1} vs {s2}: U={u_stat:.0f}, p={p_val:.4f}, '
                  f'p_corrected={p_corrected:.4f} {sig}, d={cohens_d:.3f}')
            print(f'    Subtype {s1}: mean={np.mean(g1):.2f} ± {np.std(g1, ddof=1):.2f} (n={len(g1)})')
            print(f'    Subtype {s2}: mean={np.mean(g2):.2f} ± {np.std(g2, ddof=1):.2f} (n={len(g2)})')

            pairwise_results.append({
                'comparison': f'S{s1}_vs_S{s2}', 'U': u_stat, 'p_raw': p_val,
                'p_corrected': p_corrected, 'cohens_d': cohens_d,
                'mean_1': np.mean(g1), 'mean_2': np.mean(g2),
                'n_1': len(g1), 'n_2': len(g2),
            })

    pairwise_df = pd.DataFrame(pairwise_results)
    print(f'\nSignificant pairwise comparisons (Bonferroni-corrected): '
          f'{(pairwise_df["p_corrected"] < 0.05).sum()}/{len(pairwise_df)}')
else:
    print('No clinical score columns available for pairwise testing.')

In [ ]:
# Boxplot: ADOS scores by subtype
if not test_cols:
    print('No clinical score columns with sufficient data for plotting.')
else:
    n_plot = min(len(test_cols), 4)
    fig, axes = plt.subplots(1, n_plot, figsize=(5 * n_plot, 6))
    if n_plot == 1:
        axes = [axes]

    for i, col in enumerate(test_cols[:4]):
        ax = axes[i]

        # Prepare data
        plot_data = []
        for s in sorted(asd_meta['subtype'].unique()):
            vals = asd_meta.loc[asd_meta['subtype'] == s, col].dropna()
            for v in vals:
                plot_data.append({'Subtype': f'S{s}', col: v})
        plot_df = pd.DataFrame(plot_data)

        if len(plot_df) > 0:
            sns.boxplot(data=plot_df, x='Subtype', y=col, ax=ax, palette='Set2')
            sns.stripplot(data=plot_df, x='Subtype', y=col, ax=ax, color='black',
                          alpha=0.4, size=3)

            # Add sample sizes
            for j, s in enumerate(sorted(asd_meta['subtype'].unique())):
                n = asd_meta.loc[asd_meta['subtype'] == s, col].notna().sum()
                ax.text(j, ax.get_ylim()[1], f'n={n}', ha='center', va='bottom', fontsize=8)

            ax.set_title(col.replace('_', ' ').title(), fontsize=10)
            ax.set_xlabel('Molecular Subtype')

            # Add Kruskal-Wallis p-value
            kw_row = kw_df[kw_df['score'] == col] if len(kw_df) > 0 else pd.DataFrame()
            if len(kw_row) > 0:
                p = kw_row.iloc[0]['p_value']
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                ax.set_title(f'{col.replace("_", " ").title()}\n(KW p={p:.4f} {sig})', fontsize=10)

    plt.suptitle(f'GSE111175 Blood: Clinical Scores by Molecular Subtype (k={optimal_k})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ados_by_subtype_boxplot.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Spearman correlation: Individual pathway scores vs ADOS

if primary_col:
    print(f'=== Spearman Correlation: Pathway Scores vs {primary_col} ===\n')

    corr_results = []
    for pathway in asd_scores.columns:
        score_vals = asd_scores[pathway].values
        clinical_vals = asd_meta.loc[asd_scores.index, primary_col].values
        valid = ~np.isnan(clinical_vals)

        if valid.sum() >= 10:
            rho, p_val = stats.spearmanr(score_vals[valid], clinical_vals[valid])
            corr_results.append({
                'pathway': pathway, 'rho': rho, 'p_value': p_val, 'n': int(valid.sum()),
            })

    corr_df = pd.DataFrame(corr_results).sort_values('p_value')

    # FDR correction (Benjamini-Hochberg)
    from statsmodels.stats.multitest import multipletests
    if len(corr_df) > 0:
        _, corr_df['p_fdr'], _, _ = multipletests(corr_df['p_value'], method='fdr_bh')

        print(f'{"Pathway":<30} {"rho":>6} {"p":>10} {"p_FDR":>10} {"Sig":>5}')
        print('-' * 65)
        for _, row in corr_df.iterrows():
            sig = '***' if row['p_fdr'] < 0.001 else '**' if row['p_fdr'] < 0.01 else '*' if row['p_fdr'] < 0.05 else ''
            print(f'{row["pathway"]:<30} {row["rho"]:>6.3f} {row["p_value"]:>10.4f} {row["p_fdr"]:>10.4f} {sig:>5}')

        sig_count = (corr_df['p_fdr'] < 0.05).sum()
        print(f'\nSignificant (FDR < 0.05): {sig_count}/{len(corr_df)} pathways')

        # Save correlation table
        corr_df.to_csv(os.path.join(OUTPUT_DIR, 'pathway_ados_correlation.csv'), index=False)
else:
    print('No ADOS column available for correlation analysis.')
    corr_df = pd.DataFrame()

In [ ]:
# Heatmap: Pathway-ADOS correlation
if primary_col and len(corr_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))

    # Order by absolute correlation
    plot_corr = corr_df.sort_values('rho', ascending=True)

    colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in plot_corr['p_fdr']]
    bars = ax.barh(range(len(plot_corr)), plot_corr['rho'].values, color=colors)

    ax.set_yticks(range(len(plot_corr)))
    ax.set_yticklabels([p.replace('_', ' ') for p in plot_corr['pathway']], fontsize=9)
    ax.set_xlabel(f'Spearman ρ with {primary_col.replace("_", " ").title()}')
    ax.axvline(x=0, color='black', linewidth=0.5)

    # Add significance markers
    for i, (_, row) in enumerate(plot_corr.iterrows()):
        if row['p_fdr'] < 0.05:
            marker = '***' if row['p_fdr'] < 0.001 else '**' if row['p_fdr'] < 0.01 else '*'
            x_pos = row['rho'] + 0.01 if row['rho'] >= 0 else row['rho'] - 0.04
            ax.text(x_pos, i, marker, va='center', fontsize=10, color='#e74c3c')

    ax.legend(['FDR < 0.05', 'Not significant'], loc='lower right', fontsize=9)
    plt.title(f'GSE111175: Pathway Score Correlation with {primary_col.replace("_", " ").title()}\n'
              f'(ASD samples, n={corr_df.iloc[0]["n"]})',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'pathway_ados_correlation.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Summary of clinical correlation findings
print('=' * 60)
print('CLINICAL CORRELATION SUMMARY')
print('=' * 60)

print(f'\nDataset: GSE111175 (Gazestani et al. 2019)')
print(f'ASD samples with subtype assignments: {len(asd_meta)}')

if primary_col:
    ados_available = asd_meta[primary_col].notna().sum()
    print(f'Primary clinical score: {primary_col} (n={ados_available} with data)')

# Kruskal-Wallis summary
if len(kw_df) > 0:
    print(f'\n--- Subtype Differences (Kruskal-Wallis) ---')
    for _, row in kw_df.iterrows():
        sig = 'YES' if row['p_value'] < 0.05 else 'NO'
        print(f'  {row["score"]}: H={row["H_statistic"]:.3f}, p={row["p_value"]:.4f} → Significant: {sig}')

# Correlation summary
if len(corr_df) > 0:
    print(f'\n--- Pathway-ADOS Correlation (Spearman) ---')
    sig_pathways = corr_df[corr_df['p_fdr'] < 0.05]
    print(f'  Significant correlations (FDR<0.05): {len(sig_pathways)}/{len(corr_df)}')
    if len(sig_pathways) > 0:
        for _, row in sig_pathways.iterrows():
            direction = '↑' if row['rho'] > 0 else '↓'
            print(f'    {direction} {row["pathway"]}: ρ={row["rho"]:.3f} (p_FDR={row["p_fdr"]:.4f})')

print(f'\nConclusion: Blood-based molecular subtypes ', end='')
if len(kw_df) > 0 and kw_df['significant'].any():
    print('show SIGNIFICANT differences in clinical severity scores.')
    print('This supports the biological relevance of pathway-based subtyping.')
else:
    print('do not show significant differences in ADOS scores.')
    print('This may reflect the toddler age range (1-4) where ADOS variance is limited,')
    print('or the blood-brain axis may attenuate clinical correlations.')

## 11. Blood vs Brain Comparison

**Addressing Vulnerability V7** — postmortem brain only.

Compare the pathway enrichment profiles of blood-based subtypes (this notebook)
against the brain-based subtypes from GSE28521 (Notebook 10). If the same pathways
are enriched/depleted across tissues, it supports cross-tissue convergence of
autism molecular pathology.

In [ ]:
# Load GSE28521 frontal cortex results (from Notebook 10)
# Try local research-results first, then GitHub URL
import os as _os

has_brain_data = False
local_scores = 'research-results/GSE28521/frontal-cortex/fc_pathway_scores.csv'
local_meta = 'research-results/GSE28521/frontal-cortex/fc_sample_metadata_with_subtypes.csv'

# Also try outputs from a prior Notebook 10 run
colab_scores = './outputs/gse28521/frontal_cortex/fc_pathway_scores.csv'
colab_meta = './outputs/gse28521/frontal_cortex/fc_sample_metadata_with_subtypes.csv'

brain_results_url = 'https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/examples/notebooks/outputs/gse28521/frontal_cortex/fc_pathway_scores.csv'
brain_meta_url = 'https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/examples/notebooks/outputs/gse28521/frontal_cortex/fc_sample_metadata_with_subtypes.csv'

for scores_path, meta_path, source_label in [
    (local_scores, local_meta, 'local research-results'),
    (colab_scores, colab_meta, 'local outputs'),
    (brain_results_url, brain_meta_url, 'GitHub'),
]:
    try:
        brain_scores = pd.read_csv(scores_path, index_col=0)
        brain_meta = pd.read_csv(meta_path, index_col=0)
        print(f'Loaded GSE28521 frontal cortex results from {source_label}:')
        print(f'  Brain samples: {len(brain_scores)} ({(brain_meta["diagnosis"] == "ASD").sum()} ASD)')
        print(f'  Brain pathways: {brain_scores.shape[1]}')
        print(f'  Brain subtypes: {brain_meta["subtype"].nunique()}')
        has_brain_data = True
        break
    except Exception:
        continue

if not has_brain_data:
    print('Could not load brain data from any source.')
    print('Skipping blood vs brain comparison. Run Notebook 10 first to generate outputs.')

In [ ]:
# Compare pathway profiles across tissues
if has_brain_data:
    # Find shared pathways between blood and brain analyses
    blood_pathways = set(asd_scores.columns)
    brain_pathways = set(brain_scores.columns)
    shared = sorted(blood_pathways & brain_pathways)
    print(f'Shared pathways: {len(shared)}/{len(blood_pathways)} blood, {len(brain_pathways)} brain')

    # Mean pathway scores by diagnosis in each tissue
    blood_asd_mean = asd_scores[shared].mean()
    blood_asd_std = asd_scores[shared].std()

    brain_asd_mask = brain_meta['diagnosis'] == 'ASD'
    brain_asd_mean = brain_scores.loc[brain_asd_mask, shared].mean()
    brain_asd_std = brain_scores.loc[brain_asd_mask, shared].std()

    # Spearman correlation of mean pathway profiles
    rho, p_val = stats.spearmanr(blood_asd_mean.values, brain_asd_mean.values)
    print(f'\nCross-tissue correlation of mean ASD pathway profiles:')
    print(f'  Spearman ρ = {rho:.4f}, p = {p_val:.4f}')
    sig = 'SIGNIFICANT' if p_val < 0.05 else 'not significant'
    print(f'  → {sig}')

    # Scatter plot: blood vs brain pathway means
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: mean score comparison
    ax = axes[0]
    ax.scatter(blood_asd_mean.values, brain_asd_mean.values, s=80, alpha=0.8,
               edgecolors='k', linewidth=0.5, c='#3498db')

    for i, pw in enumerate(shared):
        ax.annotate(pw.replace('_', '\n'), (blood_asd_mean.iloc[i], brain_asd_mean.iloc[i]),
                    fontsize=6, ha='center', va='bottom')

    ax.set_xlabel('Blood Mean Pathway Score (GSE111175)')
    ax.set_ylabel('Brain Mean Pathway Score (GSE28521 FC)')
    ax.set_title(f'Cross-Tissue Pathway Concordance\nSpearman ρ={rho:.3f}, p={p_val:.4f}')

    # Fit line
    z = np.polyfit(blood_asd_mean.values, brain_asd_mean.values, 1)
    x_line = np.linspace(blood_asd_mean.min(), blood_asd_mean.max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r--', alpha=0.5)

    # Right: side-by-side bar chart
    ax2 = axes[1]
    x = np.arange(len(shared))
    width = 0.35

    # Z-score normalize for visual comparison
    blood_z = (blood_asd_mean - blood_asd_mean.mean()) / blood_asd_mean.std()
    brain_z = (brain_asd_mean - brain_asd_mean.mean()) / brain_asd_mean.std()

    ax2.barh(x - width/2, blood_z.values, width, label='Blood (GSE111175)', color='#e74c3c', alpha=0.7)
    ax2.barh(x + width/2, brain_z.values, width, label='Brain (GSE28521 FC)', color='#3498db', alpha=0.7)
    ax2.set_yticks(x)
    ax2.set_yticklabels([p.replace('_', ' ') for p in shared], fontsize=8)
    ax2.set_xlabel('Z-normalized Mean Score')
    ax2.set_title('Pathway Profiles: Blood vs Brain')
    ax2.legend(fontsize=9)
    ax2.axvline(x=0, color='black', linewidth=0.5)

    plt.suptitle('Cross-Tissue Validation: Blood (GSE111175) vs Brain (GSE28521)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'blood_vs_brain_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping blood vs brain comparison (no brain data loaded).')

## 12. Benchmark Comparison

Compare the framework's pathway-based GMM approach against alternative methods.

In [ ]:
# Run benchmark comparison on ASD samples
print('Running benchmark comparison...')
bench_result = run_benchmark_comparison(
    gene_burdens=asd_expression,
    pathway_scores=asd_scores,
    pathways=pathways,
    n_clusters=optimal_k,
    seed=SEED,
)

print('\n' + bench_result.format_report())

In [ ]:
# Visualize benchmark results
methods = list(bench_result.method_results.keys())
silhouettes = [bench_result.method_results[m].silhouette for m in methods]
runtimes = [bench_result.method_results[m].runtime_seconds for m in methods]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Silhouette comparison
colors = ['#2ecc71' if m == bench_result.best_method else '#3498db' for m in methods]
bars = ax1.barh(methods, silhouettes, color=colors)
ax1.set_xlabel('Silhouette Score (higher is better)')
ax1.set_title('Clustering Quality: Method Comparison')
for bar, val in zip(bars, silhouettes):
    ax1.text(max(bar.get_width() + 0.005, 0.01), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

# Runtime comparison
ax2.barh(methods, runtimes, color='#9b59b6')
ax2.set_xlabel('Runtime (seconds)')
ax2.set_title('Computational Cost')
for bar, val in zip(ax2.patches, runtimes):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}s', va='center', fontsize=10)

plt.suptitle(f'GSE111175 Blood: Benchmark Comparison (k={optimal_k})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'benchmark_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13. Algorithm Comparison

Compare GMM against K-means, Hierarchical, and Spectral clustering on the blood dataset.

In [ ]:
# Compare all clustering algorithms
algo_comparison = compare_algorithms(
    data=asd_scores.values,
    n_clusters=optimal_k,
    seed=SEED,
)

print(f'Most stable algorithm: {algo_comparison.most_stable_algorithm}')
print(f'\nPairwise ARI (inter-algorithm agreement):')
for pair, ari in algo_comparison.pairwise_ari.items():
    print(f'  {pair}: {ari:.4f}')

print(f'\nPer-algorithm metrics:')
for algo, res in algo_comparison.results.items():
    print(f'  {algo}: silhouette={res.silhouette:.4f}, CH={res.calinski_harabasz:.1f}, '
          f'DB={res.davies_bouldin:.4f}')

## 14. Summary & Export

Save all results for downstream use and generate the final summary.

In [ ]:
# Save key outputs
asd_scores.to_csv(os.path.join(OUTPUT_DIR, 'pathway_scores_asd.csv'))
pathway_scores.to_csv(os.path.join(OUTPUT_DIR, 'pathway_scores_all.csv'))
asd_meta.to_csv(os.path.join(OUTPUT_DIR, 'sample_metadata_with_subtypes.csv'))
gene_expression.to_csv(os.path.join(OUTPUT_DIR, 'gene_expression_processed.csv'))

# Save comprehensive results JSON
import json

results_summary = {
    'dataset': 'GSE111175',
    'citation': 'Gazestani et al. 2019, Nature Neuroscience',
    'tissue': 'blood leukocytes',
    'platform': 'Illumina HumanHT-12 V4.0',
    'n_total_samples': int(len(pathway_scores)),
    'n_asd': int(asd_mask.sum()),
    'n_control': int((~asd_mask).sum()),
    'n_genes': int(gene_expression.shape[1]),
    'n_pathways_scored': int(scoring_result.n_pathways_scored),
    'scoring_method': 'ssGSEA',
    'optimal_k': int(optimal_k),
    'clustering_algorithm': 'GMM',
    'silhouette': float(clustering.silhouette),
    'calinski_harabasz': float(clustering.calinski_harabasz),
    'davies_bouldin': float(clustering.davies_bouldin),
    'validation_all_passed': bool(val_result.all_passed),
    'validation_gates': [
        {'name': str(g.name), 'passed': bool(g.passed), 'metric': str(g.metric_name),
         'value': float(g.metric_value), 'threshold': float(g.threshold)}
        for g in val_result.results
    ],
    'benchmark_best_method': str(bench_result.best_method),
    'benchmark_ranking': [str(r) for r in bench_result.ranking],
    'subtype_sizes': {str(i): int((labels == i).sum()) for i in range(optimal_k)},
    'clinical_correlation': {
        'primary_score': str(primary_col) if primary_col else None,
        'kruskal_wallis': kw_df.to_dict('records') if len(kw_df) > 0 else [],
        'pathway_correlations_fdr_sig': int((corr_df['p_fdr'] < 0.05).sum()) if len(corr_df) > 0 else 0,
    },
    'vulnerabilities_addressed': ['V5_sample_size', 'V6_clinical_correlation', 'V7_blood_tissue'],
    'framework_version': '0.3.0',
    'seed': SEED,
}

if has_brain_data:
    results_summary['cross_tissue'] = {
        'brain_dataset': 'GSE28521',
        'spearman_rho': float(rho),
        'spearman_p': float(p_val),
    }

with open(os.path.join(OUTPUT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print('\n' + '=' * 60)
print('ANALYSIS COMPLETE')
print('=' * 60)
print(f'\nDataset: GSE111175 (Gazestani et al. 2019, Nature Neuroscience)')
print(f'Tissue: Blood leukocytes (toddlers age 1-4)')
print(f'Samples: {len(pathway_scores)} total ({asd_mask.sum()} ASD, {(~asd_mask).sum()} Control)')
print(f'Genes: {gene_expression.shape[1]} → {scoring_result.n_pathways_scored} pathways (ssGSEA)')
print(f'Optimal subtypes: {optimal_k} (BIC-selected, GMM)')
print(f'Silhouette: {clustering.silhouette:.4f}')
print(f'Validation: {"ALL PASSED" if val_result.all_passed else "SOME FAILED"}')
print(f'Best method: {bench_result.best_method}')

print(f'\n--- Vulnerabilities Addressed ---')
print(f'  V5 (sample size): n={asd_mask.sum()} ASD (vs n=16 in GSE28521 FC)')
print(f'  V6 (clinical):    ADOS correlation tested ({primary_col})')
print(f'  V7 (tissue):      Blood-based subtyping (not postmortem brain)')

print(f'\nOutputs saved to: {OUTPUT_DIR}/')
print('  - pathway_scores_asd.csv / pathway_scores_all.csv')
print('  - sample_metadata_with_subtypes.csv')
print('  - gene_expression_processed.csv')
print('  - results_summary.json')
print('  - model_selection.png')
print('  - pca_scatter_subtypes.png')
print('  - subtype_heatmap.png / gene_heatmap.png')
print('  - ados_by_subtype_boxplot.png')
print('  - pathway_ados_correlation.png / pathway_ados_correlation.csv')
if has_brain_data:
    print('  - blood_vs_brain_comparison.png')
print('  - benchmark_comparison.png')

---

## References

1. Gazestani VH, et al. (2019). A perturbed gene network containing PI3K-AKT, RAS-ERK and WNT-β-catenin pathways in leukocytes is linked to ASD genetics and symptom severity. *Nature Neuroscience*, 22:1624-1634. [PMID: 31551594](https://pubmed.ncbi.nlm.nih.gov/31551594/)
2. Voineagu I, et al. (2011). Transcriptomic analysis of autistic brain reveals convergent molecular pathology. *Nature*, 474(7351):380-384. [PMID: 21614001](https://pubmed.ncbi.nlm.nih.gov/21614001/)
3. Satterstrom FK, et al. (2020). Large-Scale Exome Sequencing Study Implicates Both Developmental and Functional Changes in the Neurobiology of Autism. *Cell*, 180(3):568-584. [PMID: 31981491](https://pubmed.ncbi.nlm.nih.gov/31981491/)
4. Chauhan R (2026). Pathway Subtyping Framework v0.3.0. *Zenodo*. [DOI: 10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)

## Data Availability

- **GSE111175:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE111175
- **GSE28521:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521
- **Framework:** https://github.com/topmist-admin/pathway-subtyping-framework
- **PyPI:** `pip install pathway-subtyping`

## License

This notebook is released under CC-BY 4.0.